In [1]:
import json, re
import pandas as pd
import numpy as np
from get_electronic import get_electronic_dipole, get_electronic_quadrupole, get_stddev
from os import path
from pathlib import Path
from numpy import sign
import subprocess
from IPython.display import display

In [2]:
with open('output.json') as f:
    data = json.load(f)

In [3]:
with open('jacob_ladder.json') as f:
    ladder = json.load(f)

In [4]:
data

{'AlF': [{'name': 'AlF',
   'method': 'BLYP',
   'basis': 'aug-pc-4p',
   'method_type': 'dft',
   'method_level': 2,
   'quadrupoles': [-16.0501, -16.0501, -22.7295],
   'quadrupoles_off_diag': [-0.0, -0.0, -0.0],
   'dipoles': [-0.0, 0.0, -1.4687, 1.4687],
   'quadrupoles_elec': [-11.932880197700001,
    -11.932880197700001,
    -104.86651449752497],
   'dipoles_elec': [-0.0, 0.0, -28.71515488961, 28.71515488961],
   'std_dev': [0.736480580429654, 0.736480580429654, 1.7501494615108055],
   'spin_polarized': 0},
  {'name': 'AlF',
   'method': 'ccsdT',
   'basis': 'aug-cc-pcVQZp',
   'method_type': 'wft',
   'quadrupoles_ccsdt': [-15.765946654685989,
    -15.765946654685989,
    -22.819975731761097],
   'quadrupoles_mp2': [-15.832458569625961,
    -15.832458569625961,
    -22.936119389084194],
   'quadrupoles_ccsd': [-15.69775344747287,
    -15.69775344747287,
    -22.96806393845115],
   'quadrupoles_hf': [-15.7093, -15.7093, -23.7877],
   'dipoles_ccsdt': [0, 0, -1.4655, 1.4655],
   '

In [4]:
def search(key, value, molecule):
    result = []
    for d in data[molecule]:
        if d[key] == value:
            result.append(d)
    return result

In [5]:
def load():
    global data
    with open('output.json') as f:
        data = json.load(f)

In [6]:
def save(data):
    with open("output.json", 'w') as outfile:
        json.dump(data, outfile)
    result = subprocess.run(['sh', 'commit.sh'], stdout=subprocess.PIPE)
    print(result.stdout)
    print('File saved!')

In [7]:
def save_as(data, name):
    with open(name, 'w') as outfile:
        json.dump(data, outfile)
    print('File saved as {0}!'.format(name))

In [8]:
def retrieve(molecule, method, basis):
    d = data[molecule]
    for m in d:
        if m['method'] == method and m['basis'] == basis:
            return m
    return {}

In [9]:
def get_dft_species():
    result = []
    for v in data.values():
        for d in v:
            if d['method_type'] == 'dft':
                result.append(d['name'])
                break
    return result
def get_wft_species():
    result = []
    for v in data.values():
        for d in v:
            if d['method_type'] == 'wft':
                result.append(d['name'])
                break
    return result

In [10]:
def get_functionals():
    functionals = []
    for v in data.values():
        for n in v:
            if n['method_type'] == 'dft':
                functionals.append(n['method'])
    functionals = np.unique(functionals)
    return functionals

In [11]:
def delete(molecule, method, basis):
    for c, d in enumerate(data[molecule]):
        if d.get('method') == method and d.get('basis') == basis:
            data[molecule].pop(c)
    return

def pop(molecule, method, basis, key):
    for d in data[molecule]:
        if d.get('method') == method and d.get('basis') == basis:
            d.pop(key, None)
    return

def modify(molecule, method, basis, old, new):
    for d in data[molecule]:
        if d.get('method') == method and d.get('basis') == basis:
            d[old] = new
    return

def add(molecule, method, basis, key, value):
    for d in data[molecule]:
        if d.get('method') == method and d.get('basis') == basis:
            d[key] = value

In [12]:
from pymatgen.core.periodic_table import Element

In [13]:
def count_e(name):
    if name == 'CH2-t':
        return 8
    l = [a for a in re.split(r'([A-Z][a-z]*)', name) if a]
    c = 0
    e = 0
    prev = ''
    while c < len(l):
        if l[c].isnumeric():
            e -= Element(prev).Z
            e += Element(prev).Z * int(l[c])
        else:
            e += Element(l[c]).Z
            prev = l[c]
        c += 1
    return e

In [14]:
functionals_sorted = sorted(sorted(get_functionals(), key=lambda v: v.upper()), key=lambda x: ladder[x])
functionals_sorted

['Slater',
 'SPW92',
 'B97-D3',
 'BLYP',
 'BPBE',
 'mPW91',
 'N12',
 'PBE',
 'SOGGA11',
 'B97M-V',
 'M06-L',
 'mBEEF',
 'MGGA_MS2',
 'MN15-L',
 'revM06-L',
 'SCAN',
 'TPSS',
 'B3LYP',
 'B97-2',
 'HFLYP',
 'M06-2X',
 'PBE0',
 'M06',
 'M08-HX',
 'MN15',
 'SCAN0',
 'TPSSh',
 'CAM-B3LYP',
 'wB97X-D',
 'wB97X-V',
 'wB97M-V',
 'wM05-D']

In [15]:
def make_dft_table(key, domain=data.keys()):
    """
    Makes a table of molecules in DOMAIN of desired KEY.
    """
    values = []
    index = pd.MultiIndex.from_product([domain], names=['molecule'])
    columns = pd.MultiIndex.from_product([functionals_sorted, ['XX','YY','ZZ']], names=['method', 'component'])
    for d in domain:
        value = data[d]
        temp = [[0, 0, 0] for i in range(len(functionals_sorted))]
        for i in value:
            if i['method_type'] == 'dft':
                method = i['method']
                ind = functionals_sorted.index(method)
                temp[ind] = i[key]
        values.append(np.ravel(temp).tolist())
    df = pd.DataFrame(values, index=index, columns=columns)
    return df
              

In [16]:
quadrupoles_table = make_dft_table('quadrupoles')

In [17]:
make_dft_table('std_dev').loc['He']

method       Slater                        SPW92                      B97-D3  \
component        XX        YY        ZZ       XX       YY       ZZ        XX   
molecule                                                                       
He         0.665192  0.665192  0.665192  0.65497  0.65497  0.65497  0.641059   

method                             BLYP  ...   wB97X-D   wB97X-V            \
component        YY        ZZ        XX  ...        ZZ        XX        YY   
molecule                                 ...                                 
He         0.641059  0.641059  0.647148  ...  0.635556  0.642392  0.642392   

method                wB97M-V                     wM05-D            
component        ZZ        XX        YY        ZZ     XX   YY   ZZ  
molecule                                                            
He         0.642392  0.643086  0.643086  0.643086    0.0  0.0  0.0  

[1 rows x 96 columns]

In [21]:
make_dft_table('std_dev').loc['BeH2']

method       Slater                     SPW92              B97-D3            \
component        XX        YY        ZZ    XX   YY   ZZ        XX        YY   
molecule                                                                      
BeH2       0.947195  0.947195  1.893168   0.0  0.0  0.0  0.911522  0.911522   

method                  BLYP  ... CAM-B3LYP wB97X-D           wB97X-V       \
component       ZZ        XX  ...        ZZ      XX   YY   ZZ      XX   YY   
molecule                      ...                                            
BeH2       1.89122  0.915632  ...  1.891089     0.0  0.0  0.0     0.0  0.0   

method         wB97M-V            
component   ZZ      XX   YY   ZZ  
molecule                          
BeH2       0.0     0.0  0.0  0.0  

[1 rows x 81 columns]

In [22]:
make_dft_table('std_dev').loc['Ne']

method      Slater                      SPW92                      B97-D3  \
component       XX       YY       ZZ       XX       YY       ZZ        XX   
molecule                                                                    
Ne         0.57839  0.57839  0.57839  0.57307  0.57307  0.57307  0.567968   

method                             BLYP  ... CAM-B3LYP   wB97X-D            \
component        YY        ZZ        XX  ...        ZZ        XX        YY   
molecule                                 ...                                 
Ne         0.567968  0.567968  0.573919  ...   0.57041  0.565304  0.565304   

method                wB97X-V                       wB97M-V            \
component        ZZ        XX        YY        ZZ        XX        YY   
molecule                                                                
Ne         0.565304  0.569412  0.569412  0.569412  0.569934  0.569934   

method               
component        ZZ  
molecule             
Ne         0.569934  

[1 rows x 81 columns]

In [23]:
make_dft_table('std_dev').loc['Be']

method       Slater                     SPW92              B97-D3            \
component        XX        YY        ZZ    XX   YY   ZZ        XX        YY   
molecule                                                                      
Be         1.222163  1.222163  1.222163   0.0  0.0  0.0  1.188805  1.188805   

method              BLYP  ... CAM-B3LYP wB97X-D           wB97X-V            \
component        ZZ   XX  ...        ZZ      XX   YY   ZZ      XX   YY   ZZ   
molecule                  ...                                                 
Be         1.188805  0.0  ...  1.183775     0.0  0.0  0.0     0.0  0.0  0.0   

method    wB97M-V            
component      XX   YY   ZZ  
molecule                     
Be            0.0  0.0  0.0  

[1 rows x 81 columns]

In [24]:
make_dft_table('std_dev').loc['N']

method       Slater                     SPW92              B97-D3            \
component        XX        YY        ZZ    XX   YY   ZZ        XX        YY   
molecule                                                                      
N          0.784452  0.784452  0.784452   0.0  0.0  0.0  0.765283  0.765283   

method                   BLYP  ... CAM-B3LYP   wB97X-D                      \
component        ZZ        XX  ...        ZZ        XX        YY        ZZ   
molecule                       ...                                           
N          0.765283  0.775054  ...  0.770291  0.764623  0.764623  0.764623   

method      wB97X-V                       wB97M-V                      
component        XX        YY        ZZ        XX        YY        ZZ  
molecule                                                               
N          0.770153  0.770153  0.770153  0.773229  0.773229  0.773229  

[1 rows x 81 columns]

In [18]:
wft_method = ['hf', 'MP2', 'CCSD', 'CCSD\(T\)']
wft_basis = ['aug-cc-pcV5Zp', 'aug-cc-pcVQZp', 'aug-cc-pcVTZp', 'CBS']

In [19]:
def make_table_wft_extrap(domain=data.keys()):
    """
    Makes a table of DOMAIN of wft method of desired KEY.
    """
    values = np.array([])
    index = pd.MultiIndex.from_product([domain, ['5Z', 'QZ', 'TZ', 'CBS']], names=['molecule', 'basis'])
    columns = pd.MultiIndex.from_product([wft_method, ['XX','YY','ZZ']], names=['method', 'component'])
    for d in domain:
        m_name = d
        for b in wft_basis:
            temp = [[0, 0, 0] for k in range(len(wft_method))]
            info = search('basis', b, m_name)
            if info != []:
                temp[0] = info[0]['std_dev_hf']
                temp[1] = info[0]['std_dev_mp2']
                temp[2] = info[0]['std_dev_ccsd']
                temp[3] = info[0]['std_dev_ccsdt']
            temp = np.ravel(temp)
            values = np.vstack((values, temp)) if values.size else temp
    df = pd.DataFrame(values, index=index, columns=columns)
    return df

In [20]:
wft_table = make_table_wft_extrap(get_wft_species())

KeyError: 'std_dev_hf'

In [44]:
wft_table[wft_table.index.get_level_values('basis') == '5Z']

method               hf                          MP2                        \
component            XX       YY       ZZ         XX         YY         ZZ   
molecule basis                                                               
AlF      5Z      0.0000   0.0000   0.0000   0.000000   0.000000   0.000000   
Ar       5Z    -11.6741 -11.6741 -11.6741 -11.656129 -11.656129 -11.656129   
Be       5Z      0.0000   0.0000   0.0000   0.000000   0.000000   0.000000   
BeH      5Z      0.0000   0.0000   0.0000   0.000000   0.000000   0.000000   
BeH2     5Z      0.0000   0.0000   0.0000   0.000000   0.000000   0.000000   
...                 ...      ...      ...        ...        ...        ...   
HCl      5Z    -14.0726 -14.0726 -10.2180 -14.012628 -14.012628 -10.243776   
NH       5Z     -6.2690  -6.2690  -5.4578  -6.372837  -5.470655   0.836409   
PH       5Z    -14.4015 -14.4015 -15.4878 -14.363615 -15.049447  -0.236188   
SiO      5Z    -16.0826 -16.0826 -25.3663 -16.569470 -16.569470 -23.618388   
LiH      5Z      0.0000   0.0000   0.0000   0.000000   0.000000   0.000000   

method               CCSD                        CCSD\(T\)             \
component              XX         YY         ZZ         XX         YY   
molecule basis                                                          
AlF      5Z      0.000000   0.000000   0.000000   0.000000   0.000000   
Ar       5Z    -11.624722 -11.624722 -11.624722 -11.638576 -11.638576   
Be       5Z      0.000000   0.000000   0.000000   0.000000   0.000000   
BeH      5Z      0.000000   0.000000   0.000000   0.000000   0.000000   
BeH2     5Z      0.000000   0.000000   0.000000   0.000000   0.000000   
...                   ...        ...        ...        ...        ...   
HCl      5Z    -13.947865 -13.947865 -10.236849 -13.963534 -13.963534   
NH       5Z     -6.347012  -5.482626   0.822085  -6.365237  -5.506568   
PH       5Z    -14.302954 -15.020933  -0.233700 -14.315261 -14.995915   
SiO      5Z    -16.259103 -16.259103 -24.055255 -16.382039 -16.382039   
LiH      5Z      0.000000   0.000000   0.000000   0.000000   0.000000   

method                     
component              ZZ  
molecule basis             
AlF      5Z      0.000000  
Ar       5Z    -11.638576  
Be       5Z      0.000000  
BeH      5Z      0.000000  
BeH2     5Z      0.000000  
...                   ...  
HCl      5Z    -10.266440  
NH       5Z      0.814889  
PH       5Z     -0.229194  
SiO      5Z    -23.702654  
LiH      5Z      0.000000  

[73 rows x 12 columns]

In [494]:
wft_table.loc['C2H']

method           hf                           MP2                      \
component        XX        YY        ZZ        XX        YY        ZZ   
basis                                                                   
5Z         0.860498  0.860498  1.545005 -0.000000 -0.000000  1.383428   
QZ         0.860668  0.860668  1.545016  0.870421  0.870421  1.547620   
TZ         0.861166  0.861166  1.544931  0.871710  0.871710  1.548739   
CBS        0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   

method         CCSD                     CCSD\(T\)                      
component        XX        YY        ZZ        XX        YY        ZZ  
basis                                                                  
5Z        -0.000000 -0.000000  1.383428 -0.000000 -0.000000  1.383428  
QZ         0.855555  0.855555  1.557354  0.855798  0.855798  1.558396  
TZ         0.857163  0.857163  1.558318  0.857549  0.857549  1.559362  
CBS        0.000000  0.000000  0.000000  0.000000  0.000000  0.000000

In [341]:
save(data)

File saved!


In [87]:
data['HBS']

[{'name': 'HBS',
  'method': 'B97-2',
  'basis': 'aug-pc-4p',
  'method_type': 'dft',
  'method_level': 4,
  'quadrupoles': [-19.6484, -19.6484, -19.341],
  'quadrupoles_off_diag': [0.0, 0.0, 0.0],
  'dipoles': [-0.0, 0.0, 1.4588, 1.4588],
  'quadrupoles_elec': [-14.6081334868, -14.6081334868, -165.44134642761856],
  'dipoles_elec': [-0.0, 0.0, 46.72643396564, 46.72643396564],
  'std_dev': [0.814865674563494, 0.814865674563494, 1.734643359076025],
  'spin_polarized': 0},
 {'name': 'HBS',
  'method': 'HFLYP',
  'basis': 'aug-pc-4p',
  'method_type': 'dft',
  'method_level': 4,
  'quadrupoles': [-19.916, -19.916, -19.9894],
  'quadrupoles_off_diag': [-0.0, -0.0, 0.0],
  'dipoles': [0.0, -0.0, 2.0072, 2.0072],
  'quadrupoles_elec': [-14.807087932000002,
   -14.807087932000002,
   -165.92341691441857],
  'dipoles_elec': [0.0, -0.0, 46.94219114216, 46.94219114216],
  'std_dev': [0.8203959114077456, 0.8203959114077456, 1.728914209790152],
  'spin_polarized': 0},
 {'name': 'HBS',
  'method': 

In [90]:
#this cell block is used to modify dipole values
modify('HBS', 'ccsdT', 'aug-cc-pcVQZp', 'dipoles_c', [0, 0, 1.4791, 1.4791])

In [501]:
save(data)

b"\x1bSaving updates to GitHub...\x1b\nOn branch master\nYour branch is up to date with 'origin/master'.\n\nChanges not staged for commit:\n\tmodified:   ../geometries/cbs_limit/LiH_ccsdT_aug-cc-pcV5Z.inp\n\tmodified:   ../geometries/cbs_limit/LiH_ccsdT_aug-cc-pcVQZ.inp\n\tmodified:   ../geometries/cbs_limit/LiH_ccsdT_aug-cc-pcVTZ.inp\n\tmodified:   ../geometries/cbs_limit/Mg_ccsdT_aug-cc-pcV5Z.inp\n\tmodified:   ../geometries/cbs_limit/Na_ccsdT_aug-cc-pcV5Z.inp\n\tmodified:   ../geometries/cbs_limit/make_input.py\n\tdeleted:    ../geometries/inputs/dft/Be_B3LYP5_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_B3LYP_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_B97-D3_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_B97M-V_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_BLYP_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_BPBE_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_CAM-B3LYP_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/B

In [481]:
#use this cell block for electronic component calculation

input_path = Path.home() / "summer/research/geometries/geometries"
ang_to_bohr = 1.88973
ang2_to_bohr2 = 3.571079
debye_to_e = 0.3934303
ea02_to_debye_ang = 1.345033669
debye_ang_to_ea02 = 0.743477

def get_electronic_dipole(string, dipole):
    info = re.search('molecule\s*\d\s\d\s([\s\S]*?)\$end', string).group(1)
    x = y = z = 0
    for l in info.splitlines():
        line = l.split()
        charge = Element(re.sub('\d+', '', line[0])).Z
        if len(line) == 1:
            line += [0, 0, 0]
        x += charge * float(line[1])
        y += charge * float(line[2])
        z += charge * float(line[3])
    x_elec = dipole[0] * debye_to_e - x * ang_to_bohr
    y_elec = dipole[1] * debye_to_e - y * ang_to_bohr
    z_elec = dipole[2] * debye_to_e - z * ang_to_bohr
    total_elec = np.sqrt(x_elec ** 2 + y_elec ** 2 + z_elec ** 2)
    return [x_elec, y_elec, z_elec, total_elec]

def get_electronic_quadrupole(string, quadrupole):
    info = re.search('molecule\s*\d\s\d\s([\s\S]*?)\$end', string).group(1)
    xx = yy = zz = 0
    for l in info.splitlines():
        line = l.split()
        charge = Element(re.sub('\d+', '', line[0])).Z
        if len(line) == 1:
            line += [0, 0, 0]
        xx += charge * float(line[1]) ** 2
        yy += charge * float(line[2]) ** 2
        zz += charge * float(line[3]) ** 2
    xx_elec = quadrupole[0] * debye_ang_to_ea02 - xx * ang2_to_bohr2
    yy_elec = quadrupole[1] * debye_ang_to_ea02 - yy * ang2_to_bohr2
    zz_elec = quadrupole[2] * debye_ang_to_ea02 - zz * ang2_to_bohr2
    return [xx_elec, yy_elec, zz_elec]

def get_stddev(string, dipole, quadrupole):
    info = re.search('molecule\s*\d\s\d\s([\s\S]*?)\$end', string).group(1)
    tote = int(re.search('molecule\s*(\d\s\d\s)', string).group(1).splitlines()[0].split()[0])
    for l in info.splitlines():
        line = l.split()
        charge = Element(re.sub('\d+', '', line[0])).Z
        tote += Element(re.sub('\d+', '', line[0])).Z
    xx_std = np.sqrt((quadrupole[0]/-tote) - (dipole[0]/-tote) ** 2)
    yy_std = np.sqrt((quadrupole[1]/-tote) - (dipole[1]/-tote) ** 2)
    zz_std = np.sqrt((quadrupole[2]/-tote) - (dipole[2]/-tote) ** 2)
    return [xx_std, yy_std, zz_std]

def get_electronic_for_wft(molecule, method, basis):
    filename = input_path / "{}.xyz".format(molecule)
    with open(filename, 'r') as file:
        string = file.read()
    d = retrieve(molecule, method, basis)
    if d == {}:
        print('WFT method has not been calculated for {0} of basis {1}!'.format(molecule, basis))
        return
    add(molecule, method, basis, 'dipoles_elec_hf', get_electronic_dipole(string, d['dipoles_hf']))
    add(molecule, method, basis, 'dipoles_elec_mp2', get_electronic_dipole(string, d['dipoles_mp2']))
    add(molecule, method, basis, 'dipoles_elec_ccsd', get_electronic_dipole(string, d['dipoles_ccsd']))
    add(molecule, method, basis, 'dipoles_elec_ccsdt', get_electronic_dipole(string, d['dipoles_ccsdt']))
    add(molecule, method, basis, 'quadrupoles_elec_hf', get_electronic_quadrupole(string, d['quadrupoles_hf']))
    add(molecule, method, basis, 'quadrupoles_elec_mp2', get_electronic_quadrupole(string, d['quadrupoles_mp2']))
    add(molecule, method, basis, 'quadrupoles_elec_ccsd', get_electronic_quadrupole(string, d['quadrupoles_ccsd']))
    add(molecule, method, basis, 'quadrupoles_elec_ccsdt', get_electronic_quadrupole(string, d['quadrupoles_ccsdt']))
    add(molecule, method, basis, 'std_dev_hf', get_stddev(string, d['dipoles_elec_hf'], d['quadrupoles_elec_hf']))
    add(molecule, method, basis, 'std_dev_mp2', get_stddev(string, d['dipoles_elec_mp2'], d['quadrupoles_elec_mp2']))
    add(molecule, method, basis, 'std_dev_ccsd', get_stddev(string, d['dipoles_elec_ccsd'], d['quadrupoles_elec_ccsd']))
    add(molecule, method, basis, 'std_dev_ccsdt', get_stddev(string, d['dipoles_elec_ccsdt'], d['quadrupoles_elec_ccsdt']))
    print('Electronic part successfully calculated for {0} of basis {1}!'.format(molecule, basis))

def get_electronic_for_hf(molecule, method, basis):
    filename = input_path / "{}.xyz".format(molecule)
    with open(filename, 'r') as file:
        string = file.read()
    d = retrieve(molecule, method, basis)
    if d == {}:
        print('HF method has not been calculated for {0} of basis {1}!'.format(molecule, basis))
        return
    add(molecule, method, basis, 'dipoles_elec_hf', get_electronic_dipole(string, d['dipoles_hf']))
    add(molecule, method, basis, 'dipoles_elec_mp2', [0, 0, 0])
    add(molecule, method, basis, 'dipoles_elec_ccsd', [0, 0, 0])
    add(molecule, method, basis, 'dipoles_elec_ccsdt', [0, 0, 0])
    add(molecule, method, basis, 'quadrupoles_elec_hf', get_electronic_quadrupole(string, d['quadrupoles_hf']))
    add(molecule, method, basis, 'quadrupoles_elec_mp2', [0, 0, 0])
    add(molecule, method, basis, 'quadrupoles_elec_ccsd', [0, 0, 0])
    add(molecule, method, basis, 'quadrupoles_elec_ccsdt', [0, 0, 0])
    add(molecule, method, basis, 'std_dev_hf', get_stddev(string, d['dipoles_elec_hf'], d['quadrupoles_elec_hf']))
    add(molecule, method, basis, 'std_dev_mp2', [0, 0, 0])
    add(molecule, method, basis, 'std_dev_ccsd', [0, 0, 0])
    add(molecule, method, basis, 'std_dev_ccsdt', [0, 0, 0])
    print('Electronic part successfully calculated for {0} of basis {1}!'.format(molecule, basis))
    
def get_electronic_for_dft(molecule, method, basis):
    filename = input_path / "{}.xyz".format(molecule)
    with open(filename, 'r') as file:
        string = file.read()
    d = retrieve(molecule, method, basis)
    if d == {}:
        print('DFT method has not been calculated for {0} of method {1}!'.format(molecule, method))
        return
    add(molecule, method, basis, 'dipoles_elec', get_electronic_dipole(string, d['dipoles']))
    add(molecule, method, basis, 'quadrupoles_elec', get_electronic_quadrupole(string, d['quadrupoles']))
    add(molecule, method, basis, 'std_dev', get_stddev(string, d['dipoles_elec'], d['quadrupoles_elec']))
    print('Electronic part successfully calculated for {0} of method {1}!'.format(molecule, method))

In [205]:
get_electronic_for_wft('NH', 'ccsdT', 'aug-cc-pcV5Zp')

Electronic part successfully calculated for NH of basis aug-cc-pcV5Zp!


In [453]:
mols = data.keys()

In [460]:
bases = ['aug-cc-pcV5Zp', 'aug-cc-pcVQZp', 'aug-cc-pcVTZp']
for m in mols:
    for b in bases:
        get_electronic_for_wft(m, 'ccsdT', b)

WFT method has not been calculated for AlF of basis aug-cc-pcV5Zp!
Electronic part successfully calculated for AlF of basis aug-cc-pcVQZp!
Electronic part successfully calculated for AlF of basis aug-cc-pcVTZp!
Electronic part successfully calculated for Ar of basis aug-cc-pcV5Zp!
Electronic part successfully calculated for Ar of basis aug-cc-pcVQZp!
Electronic part successfully calculated for Ar of basis aug-cc-pcVTZp!
WFT method has not been calculated for BeH of basis aug-cc-pcV5Zp!
Electronic part successfully calculated for BeH of basis aug-cc-pcVQZp!
Electronic part successfully calculated for BeH of basis aug-cc-pcVTZp!
WFT method has not been calculated for BeH2 of basis aug-cc-pcV5Zp!
Electronic part successfully calculated for BeH2 of basis aug-cc-pcVQZp!
Electronic part successfully calculated for BeH2 of basis aug-cc-pcVTZp!
Electronic part successfully calculated for BF of basis aug-cc-pcV5Zp!
Electronic part successfully calculated for BF of basis aug-cc-pcVQZp!
Electroni

In [504]:
bases = ['aug-cc-pcV5Zp', 'aug-cc-pcVQZp', 'aug-cc-pcVTZp']
for m in mols:
    for b in bases:
        get_electronic_for_hf(m, 'hf', b)

HF method has not been calculated for AlF of basis aug-cc-pcV5Zp!
HF method has not been calculated for AlF of basis aug-cc-pcVQZp!
HF method has not been calculated for AlF of basis aug-cc-pcVTZp!
HF method has not been calculated for Ar of basis aug-cc-pcV5Zp!
HF method has not been calculated for Ar of basis aug-cc-pcVQZp!
HF method has not been calculated for Ar of basis aug-cc-pcVTZp!
HF method has not been calculated for BeH of basis aug-cc-pcV5Zp!
HF method has not been calculated for BeH of basis aug-cc-pcVQZp!
HF method has not been calculated for BeH of basis aug-cc-pcVTZp!
HF method has not been calculated for BeH2 of basis aug-cc-pcV5Zp!
HF method has not been calculated for BeH2 of basis aug-cc-pcVQZp!
HF method has not been calculated for BeH2 of basis aug-cc-pcVTZp!
HF method has not been calculated for BF of basis aug-cc-pcV5Zp!
HF method has not been calculated for BF of basis aug-cc-pcVQZp!
HF method has not been calculated for BF of basis aug-cc-pcVTZp!
HF method has

In [461]:
functionals = get_functionals()
for m in mols:
    for f in functionals:
        get_electronic_for_dft(m, f, 'aug-pc-4p')

Electronic part successfully calculated for AlF of method B3LYP!
Electronic part successfully calculated for AlF of method B97-D3!
Electronic part successfully calculated for AlF of method B97M-V!
Electronic part successfully calculated for AlF of method BLYP!
Electronic part successfully calculated for AlF of method BPBE!
Electronic part successfully calculated for AlF of method CAM-B3LYP!
Electronic part successfully calculated for AlF of method M06-2X!
Electronic part successfully calculated for AlF of method M06-L!
Electronic part successfully calculated for AlF of method M08-HX!
Electronic part successfully calculated for AlF of method MGGA_MS2!
Electronic part successfully calculated for AlF of method MN15!
Electronic part successfully calculated for AlF of method MN15-L!
Electronic part successfully calculated for AlF of method PBE!
Electronic part successfully calculated for AlF of method PBE0!
Electronic part successfully calculated for AlF of method SCAN!
Electronic part succ

Electronic part successfully calculated for H of method MGGA_MS2!
Electronic part successfully calculated for H of method MN15!
Electronic part successfully calculated for H of method MN15-L!
Electronic part successfully calculated for H of method PBE!
Electronic part successfully calculated for H of method PBE0!
Electronic part successfully calculated for H of method SCAN!
Electronic part successfully calculated for H of method SCAN0!
Electronic part successfully calculated for H of method SPW92!
Electronic part successfully calculated for H of method Slater!
Electronic part successfully calculated for H of method TPSS!
Electronic part successfully calculated for H of method mBEEF!
Electronic part successfully calculated for H of method mPW91!
Electronic part successfully calculated for H of method revM06-L!
Electronic part successfully calculated for H of method wB97M-V!
Electronic part successfully calculated for H of method wB97X-D!
Electronic part successfully calculated for H of 

Electronic part successfully calculated for O3 of method CAM-B3LYP!
Electronic part successfully calculated for O3 of method M06-2X!
Electronic part successfully calculated for O3 of method M06-L!
Electronic part successfully calculated for O3 of method M08-HX!
Electronic part successfully calculated for O3 of method MGGA_MS2!
Electronic part successfully calculated for O3 of method MN15!
Electronic part successfully calculated for O3 of method MN15-L!
Electronic part successfully calculated for O3 of method PBE!
Electronic part successfully calculated for O3 of method PBE0!
Electronic part successfully calculated for O3 of method SCAN!
Electronic part successfully calculated for O3 of method SCAN0!
Electronic part successfully calculated for O3 of method SPW92!
Electronic part successfully calculated for O3 of method Slater!
Electronic part successfully calculated for O3 of method TPSS!
Electronic part successfully calculated for O3 of method mBEEF!
Electronic part successfully calcul

In [32]:
data['BF']

[{'name': 'BF',
  'method': 'BLYP',
  'basis': 'aug-pc-4p',
  'method_type': 'dft',
  'method_level': 2,
  'quadrupoles': [-10.8268, -10.8268, -12.6137],
  'quadrupoles_off_diag': [0.0, -0.0, -0.0],
  'dipoles': [-0.0, -0.0, 1.0097, 1.0097],
  'quadrupoles_elec': [-8.049476783600001,
   -8.049476783600001,
   -60.9633764850087],
  'dipoles_elec': [-0.0, -0.0, -21.14964385909, 21.14964385909],
  'std_dev': [0.7582629012420429, 0.7582629012420429, 1.4395645519048152],
  'spin_polarized': 0},
 {'name': 'BF',
  'method': 'B97-D3',
  'basis': 'aug-pc-4p',
  'method_type': 'dft',
  'method_level': 2,
  'quadrupoles': [-10.6846, -10.6846, -12.5204],
  'quadrupoles_off_diag': [-0.0, -0.0, -0.0],
  'dipoles': [0.0, -0.0, 1.0262, 1.0262],
  'quadrupoles_elec': [-7.9437543542, -7.9437543542, -60.89401008090871],
  'dipoles_elec': [0.0, -0.0, -21.14315225914, 21.14315225914],
  'std_dev': [0.7532669017685564, 0.7532669017685564, 1.4383296251589635],
  'spin_polarized': 0},
 {'name': 'BF',
  'metho

In [35]:
save(data)

b'\x1bSaving updates to GitHub...\x1b\n[master 6320558] save Mon, Aug  3, 2020  9:38:26 PM\n 672 files changed, 440695 insertions(+), 11108 deletions(-)\n create mode 100644 output/dft/20-07-28/BO_PBE0_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/BO_SCAN0_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/BO_SCAN_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/BO_SPW92_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/BS_PBE0_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/BS_PBE_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/BS_SCAN0_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/BS_SCAN_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/BS_SPW92_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/BS_Slater_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/CH3_PBE0_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/CH3_PBE_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/CH3_SCAN0_aug-pc-4p.out\n create mode 100644

In [508]:
wft_table.to_csv('wft_table_2.csv')

In [478]:
std_dev_dft = make_dft_table('std_dev', get_wft_species())

In [479]:
std_dev_dft.to_csv('dft_table.csv')

In [447]:
quadrupoles_table.loc['HCl']

method      Slater                      SPW92                   B97-D3       \
component       XX       YY       ZZ       XX       YY       ZZ     XX   YY   
molecule                                                                      
HCl       -14.4753 -14.4753 -10.7091 -14.1432 -14.1432 -10.4192    0.0  0.0   

method         BLYP  ... CAM-B3LYP wB97X-D           wB97X-V            \
component   ZZ   XX  ...        ZZ      XX   YY   ZZ      XX   YY   ZZ   
molecule             ...                                                 
HCl        0.0  0.0  ...       0.0     0.0  0.0  0.0     0.0  0.0  0.0   

method    wB97M-V            
component      XX   YY   ZZ  
molecule                     
HCl           0.0  0.0  0.0  

[1 rows x 81 columns]

In [488]:
data['HBO']

[{'name': 'HBO',
  'method': 'PBE',
  'basis': 'aug-pc-4p',
  'method_type': 'dft',
  'method_level': 2,
  'quadrupoles': [-11.0978, -11.0978, -19.9502],
  'quadrupoles_off_diag': [0.0, 0.0, 0.0],
  'dipoles': [-0.0, 0.0, -2.5855, 2.5855],
  'quadrupoles_elec': [-8.2509590506, -8.2509590506, -199.033869166212],
  'dipoles_elec': [-0.0, 0.0, -47.803149380650005, 47.803149380650005],
  'std_dev': [0.7676940913541018, 0.7676940913541018, 1.599319175331429],
  'spin_polarized': 0},
 {'name': 'HBO',
  'method': 'B97-D3',
  'basis': 'aug-pc-4p',
  'method_type': 'dft',
  'method_level': 2,
  'quadrupoles': [-11.0084, -11.0084, -20.1871],
  'quadrupoles_off_diag': [0.0, -0.0, 0.0],
  'dipoles': [-0.0, -0.0, -2.6504, 2.6504],
  'quadrupoles_elec': [-8.1844922068, -8.1844922068, -199.209998867512],
  'dipoles_elec': [-0.0, -0.0, -47.82868300712, 47.82868300712],
  'std_dev': [0.7645957011388437, 0.7645957011388437, 1.599357437857018],
  'spin_polarized': 0},
 {'name': 'HBO',
  'method': 'CAM-B3

In [507]:
save(data)

b'\x1bSaving updates to GitHub...\x1b\n[master 8bc6862] save\n 1 file changed, 1 insertion(+), 1 deletion(-)\n'
File saved!


In [557]:
wft_table.loc[('Ar', 'CBS'), ('hf', 'XX')] = 0.694399

In [567]:
wft_table.index[wft_table.index.get_level_values('basis') == 'CBS']

MultiIndex([(   'Ar', 'CBS'),
            (   'Be', 'CBS'),
            (  'BeH', 'CBS'),
            ( 'BeH2', 'CBS'),
            (   'BF', 'CBS'),
            (  'BH3', 'CBS'),
            (   'BO', 'CBS'),
            (   'BS', 'CBS'),
            (  'C2H', 'CBS'),
            ( 'C2H2', 'CBS'),
            (  'CH3', 'CBS'),
            (  'CH4', 'CBS'),
            (   'CN', 'CBS'),
            (   'CO', 'CBS'),
            (   'F2', 'CBS'),
            (    'H', 'CBS'),
            (   'H2', 'CBS'),
            (  'H2O', 'CBS'),
            (  'HBO', 'CBS'),
            (  'HCN', 'CBS'),
            (   'He', 'CBS'),
            (   'HF', 'CBS'),
            (  'HNC', 'CBS'),
            ('LiBH4', 'CBS'),
            (   'Mg', 'CBS'),
            (    'N', 'CBS'),
            (   'N2', 'CBS'),
            (   'Na', 'CBS'),
            (   'Ne', 'CBS'),
            (  'NH3', 'CBS'),
            ( 'NH3O', 'CBS'),
            (   'O2', 'CBS'),
            (   'OF', 'CBS'),
          

In [192]:
#make extrapolation in this cell
def extrapolate(dataframe):
    molecules = wft_table.index[wft_table.index.get_level_values('basis') == 'CBS']
    columns = wft_table.columns
    for m in molecules:
        for c in columns:
            if c[0] == 'hf':
                if dataframe.loc[(m[0], '5Z'), c] != 0:
                    dataframe.loc[m, c] = dataframe.loc[(m[0], '5Z'), c]
                else:
                    dataframe.loc[m, c] = dataframe.loc[(m[0], 'QZ'), c]
            if dataframe.loc[(m[0], '5Z'), c] != 0:
                dataframe.loc[m, c] = ((dataframe.loc[(m[0], '5Z'), c] - dataframe.loc[(m[0], '5Z'), ('hf', c[1])]) * 125
                                       - (dataframe.loc[(m[0], 'QZ'), c] - dataframe.loc[(m[0], 'QZ'), ('hf', c[1])]) * 64) \
                                       / (125 - 64) + dataframe.loc[(m[0], 'CBS'), ('hf', c[1])]
            else:
                dataframe.loc[m, c] = ((dataframe.loc[(m[0], 'QZ'), c] - dataframe.loc[(m[0], 'QZ'), ('hf', c[1])]) * 64
                                       - (dataframe.loc[(m[0], 'TZ'), c] - dataframe.loc[(m[0], 'TZ'), ('hf', c[1])]) * 27) \
                                       / (64 - 27) + dataframe.loc[(m[0], 'CBS'), ('hf', c[1])]
    display(dataframe)

In [506]:
extrapolate(wft_table)

method                hf                           MP2                      \
component             XX        YY        ZZ        XX        YY        ZZ   
molecule basis                                                               
AlF      5Z     0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
         QZ     0.728620  0.728620  1.762230  0.731470  0.731470  1.752314   
         TZ     0.729178  0.729178  1.762396  0.733437  0.733437  1.753416   
         CBS    0.728620  0.728620  1.762230  0.730442  0.730442  1.751630   
Ar       5Z     0.694399  0.694399  0.694399  0.693865  0.693865  0.693865   
...                  ...       ...       ...       ...       ...       ...   
SiO      CBS    0.737226  0.737226  1.611787  0.759922  0.759922  1.664609   
LiH      5Z     0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
         QZ     0.987146  0.987146  1.651777  0.988522  0.988522  1.646651   
         TZ     0.988021  0.988021  1.651583  0.991225  0.991225  1.546393   
         CBS    0.987146  0.987146  1.651777  0.987189  0.987189  1.719672   

method              CCSD                     CCSD\(T\)                      
component             XX        YY        ZZ        XX        YY        ZZ  
molecule basis                                                              
AlF      5Z     0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  
         QZ     0.728352  0.728352  1.752450  0.729932  0.729932  1.751065  
         TZ     0.730298  0.730298  1.753317  0.731928  0.731928  1.752099  
         CBS    0.727339  0.727339  1.751938  0.728883  0.728883  1.750432  
Ar       5Z     0.692929  0.692929  0.692929  0.693342  0.693342  0.693342  
...                  ...       ...       ...       ...       ...       ...  
SiO      CBS    0.745493  0.745493  1.673833  0.751225  0.751225  1.666390  
LiH      5Z     0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  
         QZ     0.996375  0.996375  1.645965  0.996507  0.996507  1.645823  
         TZ     0.999088  0.999088  1.536849  0.999238  0.999238  1.540299  
         CBS    0.995035  0.995035  1.725448  0.995153  0.995153  1.722685  

[208 rows x 12 columns]

In [507]:
wft_table.loc['C2H']

method           hf                           MP2                      \
component        XX        YY        ZZ        XX        YY        ZZ   
basis                                                                   
5Z         0.860498  0.860498  1.545005  0.000000  0.000000  0.000000   
QZ         0.860668  0.860668  1.545016  0.870421  0.870421  1.547620   
TZ         0.861166  0.861166  1.544931  0.871710  0.871710  1.548739   
CBS        0.860498  0.860498  1.545005  0.869674  0.869674  1.546731   

method         CCSD                     CCSD\(T\)                      
component        XX        YY        ZZ        XX        YY        ZZ  
basis                                                                  
5Z         0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  
QZ         0.855555  0.855555  1.557354  0.855798  0.855798  1.558396  
TZ         0.857163  0.857163  1.558318  0.857549  0.857549  1.559362  
CBS        0.854576  0.854576  1.556578  0.854714  0.854714  1.557618

In [197]:
save(data)

b"\x1bSaving updates to GitHub...\x1b\nOn branch master\nYour branch is up to date with 'origin/master'.\n\nChanges not staged for commit:\n\tmodified:   ../geometries/cbs_limit/make_input.py\n\tmodified:   .ipynb_checkpoints/ouput-checkpoint.ipynb\n\tmodified:   ouput.ipynb\n\tmodified:   output.json\n\tmodified:   wft_table_2.csv\n\nUntracked files:\n\t../geometries/cbs_limit/AlF_ccsdT_aug-cc-pcV5Z.inp\n\t../geometries/cbs_limit/AlF_ccsdT_aug-cc-pcVQZ.inp\n\t../geometries/cbs_limit/AlF_ccsdT_aug-cc-pcVTZ.inp\n\t../geometries/cbs_limit/BH2F_ccsdT_aug-cc-pcV5Z.inp\n\t../geometries/cbs_limit/BH2F_ccsdT_aug-cc-pcVQZ.inp\n\t../geometries/cbs_limit/BH2F_ccsdT_aug-cc-pcVTZ.inp\n\t../geometries/cbs_limit/BH2_ccsdT_aug-cc-pcV5Z.inp\n\t../geometries/cbs_limit/BH2_ccsdT_aug-cc-pcVQZ.inp\n\t../geometries/cbs_limit/BH2_ccsdT_aug-cc-pcVTZ.inp\n\t../geometries/cbs_limit/C2H3_ccsdT_aug-cc-pcV5Z.inp\n\t../geometries/cbs_limit/C2H3_ccsdT_aug-cc-pcVQZ.inp\n\t../geometries/cbs_limit/C2H3_ccsdT_aug-cc-pc

In [231]:
len(get_dft_species())

53

In [210]:
get_functionals()

array(['B3LYP', 'B3LYP5', 'B97-D3', 'B97M-V', 'B97M-rV', 'BLYP', 'BPBE',
       'CAM-B3LYP', 'M06-2X', 'M06-L', 'M08-HX', 'MGGA_MS2', 'MN15',
       'MN15-L', 'PBE', 'PBE0', 'SCAN', 'SCAN0', 'SPW92', 'Slater',
       'TPSS', 'mBEEF', 'mPW91', 'revM06-L', 'wB97M-V', 'wB97X-D',
       'wB97X-V'], dtype='<U9')

In [469]:
wft_table.loc['BeH2']

method           hf                           MP2                      \
component        XX        YY        ZZ        XX        YY        ZZ   
basis                                                                   
5Z         0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
QZ         0.901702  0.901702  1.900126  0.900672  0.900672  1.891046   
TZ         0.902052  0.902052  1.900504  0.902234  0.902234  1.893160   
CBS        0.901702  0.901702  1.900126  0.899787  0.899787  1.889779   

method         CCSD                     CCSD\(T\)                      
component        XX        YY        ZZ        XX        YY        ZZ  
basis                                                                  
5Z         0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  
QZ         0.901476  0.901476  1.888265  0.902072  0.902072  1.887460  
TZ         0.903037  0.903037  1.890233  0.903637  0.903637  1.889498  
CBS        0.900593  0.900593  1.887106  0.901186  0.901186  1.886248

In [245]:
make_dft_table('std_dev').loc['CH3']

method       Slater                         SPW92                     B97-D3  \
component        XX        YY        ZZ        XX        YY        ZZ     XX   
molecule                                                                       
CH3        1.150866  1.150846  0.895335  1.140646  1.140626  0.879825    0.0   

method                   BLYP  ... CAM-B3LYP   wB97X-D                     \
component   YY   ZZ        XX  ...        ZZ        XX        YY       ZZ   
molecule                       ...                                          
CH3        0.0  0.0  1.142953  ...       0.0  1.134273  1.134253  0.86577   

method    wB97X-V           wB97M-V            
component      XX   YY   ZZ      XX   YY   ZZ  
molecule                                       
CH3           0.0  0.0  0.0     0.0  0.0  0.0  

[1 rows x 81 columns]

In [248]:
len(['BHF2', 'BN', 'C2H4', 'CH3Cl', 'Cl2', 'ClCN', 'CS', 'CSO', 'HBS', 'HCCCl', 'HCHS', 'HCP', 'Li', 'Li2', 'LiCl', 'LiCN', 'NaH', 'NaLi', 'NOCl', 'NP', 'OCl2', 'P2', 'PH2', 'PH3', 'PH3O', 'S2', 'SCl2', 'SiH3', 'SiH3Cl', 'SiH3F', 'SiH4', 'SO2', 'SO-trip', 'OF2', 'SH2']
)

35

In [249]:
save(data)

b"\x1bSaving updates to GitHub...\x1b\nOn branch master\nYour branch is up to date with 'origin/master'.\n\nChanges not staged for commit:\n\tmodified:   ../geometries/cbs_limit/make_input.py\n\tdeleted:    ../geometries/inputs/dft/Be_B3LYP5_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_B3LYP_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_B97-D3_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_B97M-V_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_BLYP_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_BPBE_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_CAM-B3LYP_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_M06-2X_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_M06-L_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_M08-HX_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_MGGA_MS2_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_MN15-L_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_MN15_aug-p

In [254]:
data['NH']

[{'name': 'NH',
  'method': 'ccsdT',
  'basis': 'aug-cc-pcVDZp',
  'method_type': 'wft',
  'quadrupoles_ccsdt': [-6.450983231570466,
   -5.719957432479677,
   0.798411985892379],
  'quadrupoles_mp2': [-6.443585546400087,
   -5.659430917384136,
   0.8269939513820369],
  'quadrupoles_ccsd': [-6.435649847765377,
   -5.693460269206112,
   0.8047336441340982],
  'quadrupoles_hf': [-6.2876, -6.2876, -5.5205],
  'dipoles_ccsdt': [],
  'dipoles_mp2': [],
  'dipoles_ccsd': [],
  'dipoles_hf': [0.0, 0.0, 1.6421, 1.6421],
  'spin_polarized': 1},
 {'name': 'NH',
  'method': 'ccsdT',
  'basis': 'aug-cc-pcVQZp',
  'method_type': 'wft',
  'quadrupoles_ccsdt': [-6.373374788851031,
   -6.373374788851031,
   -5.517529865318849],
  'quadrupoles_mp2': [-6.380570719002153,
   -6.380570719002153,
   -5.48289524834353],
  'quadrupoles_ccsd': [-6.355284086035276,
   -6.355284086035276,
   -5.493252007591618],
  'quadrupoles_hf': [-6.2725, -6.2725, -5.4586],
  'dipoles_ccsdt': [0, 0, 1.5363, 1.5363],
  'dipole

In [261]:
data['C2H']

[{'name': 'C2H',
  'method': 'B97M-V',
  'basis': 'aug-pc-4p',
  'method_type': 'dft',
  'method_level': 3,
  'quadrupoles': [-12.9108, -12.9108, -9.0109],
  'quadrupoles_off_diag': [0.0, 0.0, -0.0],
  'dipoles': [-0.0, 0.0, 0.762, 0.762],
  'quadrupoles_elec': [-9.5988828516, -9.5988828516, -42.31875209981425],
  'dipoles_elec': [-0.0, 0.0, 12.115330713599999, 12.115330713599999],
  'std_dev': [0.8592878469809933, 0.8592878469809933, 1.5449144266868384],
  'spin_polarized': 1},
 {'name': 'C2H',
  'method': 'M08-HX',
  'basis': 'aug-pc-4p',
  'method_type': 'dft',
  'method_level': 5,
  'quadrupoles': [-12.9854, -12.9854, -8.9589],
  'quadrupoles_off_diag': [0.0, 0.0, 0.0],
  'dipoles': [-0.0, -0.0, 0.7373, 0.7373],
  'quadrupoles_elec': [-9.6543462358, -9.6543462358, -42.28009129581425],
  'dipoles_elec': [-0.0, -0.0, 12.10561298519, 12.10561298519],
  'std_dev': [0.8617668003227216, 0.8617668003227216, 1.5444026077365125],
  'spin_polarized': 1},
 {'name': 'C2H',
  'method': 'mBEEF',

In [264]:
forbidden_species =  ['CH2F', 'CH2NH', 'CH3BH2', 'CH3O', 'FCO', 'HCO', 'HNO', 'HNS', 'HO2', 'N2H2','N2H4', 'NH2Cl', 'NH2F', 'NH2OH', 'S2H2', 'FH-OH', 'CH3OH', 'HCOOH', 'C2H3', 'CH3NH2', 'CH3SH', 'HCONH2', 'HOCl', 'HOF', 'HOOH', 'NaCN', 'P2H4', 'Ph2OH', 'HCHO', 'FNO'] + ['Be', 'OH', 'SH', 'OF', 'SF', 'NCO', 'NO', 'OCl', 'PS', 'SCl']

In [285]:
for f in forbidden_species:
    data.pop(f, None)

In [276]:
#Check for uncompleted calculations:
get_functionals()

array(['B3LYP', 'B97-D3', 'B97M-V', 'BLYP', 'BPBE', 'CAM-B3LYP', 'M06-2X',
       'M06-L', 'M08-HX', 'MGGA_MS2', 'MN15', 'MN15-L', 'PBE', 'PBE0',
       'SCAN', 'SCAN0', 'SPW92', 'Slater', 'TPSS', 'mBEEF', 'mPW91',
       'revM06-L', 'wB97M-V', 'wB97X-D', 'wB97X-V'], dtype='<U9')

In [29]:
missing = []
for m in get_dft_species():
    for f in get_functionals():
        temp = retrieve(m, f, 'aug-pc-4p')
        if temp == {}:
            missing.append((m, f))

In [30]:
missing

[('AlF', 'B97-2'),
 ('AlF', 'HFLYP'),
 ('AlF', 'M06'),
 ('AlF', 'N12'),
 ('AlF', 'SOGGA11'),
 ('AlF', 'TPSSh'),
 ('AlF', 'wB97X-D'),
 ('AlF', 'wM05-D'),
 ('Ar', 'B97-2'),
 ('Ar', 'HFLYP'),
 ('Ar', 'M06'),
 ('Ar', 'N12'),
 ('Ar', 'SOGGA11'),
 ('Ar', 'TPSSh'),
 ('Ar', 'wM05-D'),
 ('BeH', 'B97-2'),
 ('BeH', 'HFLYP'),
 ('BeH', 'M06'),
 ('BeH', 'N12'),
 ('BeH', 'SOGGA11'),
 ('BeH', 'TPSSh'),
 ('BeH', 'wM05-D'),
 ('BeH2', 'B97-2'),
 ('BeH2', 'HFLYP'),
 ('BeH2', 'M06'),
 ('BeH2', 'N12'),
 ('BeH2', 'SOGGA11'),
 ('BeH2', 'TPSSh'),
 ('BeH2', 'wM05-D'),
 ('BF', 'B97-2'),
 ('BF', 'HFLYP'),
 ('BF', 'M06'),
 ('BF', 'N12'),
 ('BF', 'SOGGA11'),
 ('BF', 'TPSSh'),
 ('BF', 'wM05-D'),
 ('BH2', 'B97-2'),
 ('BH2', 'B97M-V'),
 ('BH2', 'HFLYP'),
 ('BH2', 'M06'),
 ('BH2', 'N12'),
 ('BH2', 'SOGGA11'),
 ('BH2', 'TPSSh'),
 ('BH2', 'wM05-D'),
 ('BH2F', 'B97-2'),
 ('BH2F', 'HFLYP'),
 ('BH2F', 'M06'),
 ('BH2F', 'N12'),
 ('BH2F', 'SOGGA11'),
 ('BH2F', 'TPSSh'),
 ('BH2F', 'wB97M-V'),
 ('BH2F', 'wB97X-V'),
 ('BH2F', 'w

In [24]:
data['PH']

[{'name': 'PH',
  'method': 'ccsdT',
  'basis': 'aug-cc-pcVDZp',
  'method_type': 'wft',
  'quadrupoles_ccsdt': [-14.561267248715938,
   -15.338024192513723,
   -0.2209217803018111],
  'quadrupoles_mp2': [-14.571018742906253,
   -15.438699962605915,
   -0.24714993662817875],
  'quadrupoles_ccsd': [-14.543109294306738,
   -15.335603132282627,
   -0.22462062276753794],
  'quadrupoles_hf': [-14.4571, -14.4571, -15.6531],
  'dipoles_ccsdt': [],
  'dipoles_mp2': [],
  'dipoles_ccsd': [],
  'dipoles_hf': [-0.0, 0.0, -0.5511, 0.5511],
  'spin_polarized': 1},
 {'name': 'PH',
  'method': 'ccsdT',
  'basis': 'aug-cc-pcVQZp',
  'method_type': 'wft',
  'quadrupoles_ccsdt': [-14.333822055296505,
   -14.333822055296505,
   -15.009566971082664],
  'quadrupoles_mp2': [-14.382714029339395,
   -14.382714029339395,
   -15.064915106506376],
  'quadrupoles_ccsd': [-14.320640725587662,
   -14.320640725587662,
   -15.03263429807314],
  'quadrupoles_hf': [-14.4047, -14.4047, -15.4891],
  'dipoles_ccsdt': [0, 

In [287]:
save(data)

b"\x1bSaving updates to GitHub...\x1b\nOn branch master\nYour branch is up to date with 'origin/master'.\n\nChanges not staged for commit:\n\tmodified:   ../geometries/cbs_limit/make_input.py\n\tdeleted:    ../geometries/inputs/dft/Be_B3LYP5_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_B3LYP_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_B97-D3_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_B97M-V_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_BLYP_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_BPBE_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_CAM-B3LYP_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_M06-2X_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_M06-L_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_M08-HX_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_MGGA_MS2_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_MN15-L_aug-pc-4.inp\n\tdeleted:    ../geometries/inputs/dft/Be_MN15_aug-p

In [74]:
missing_wft = []
for m in get_dft_species():
    temp = retrieve(m, 'ccsdT', 'aug-cc-pcV5Zp')
    if temp == {}:
        missing_wft.append(m)

In [75]:
missing_wft

['AlF',
 'BH2F',
 'BO',
 'C2H',
 'C2H2',
 'CH2BH',
 'CH3',
 'CH4',
 'ClF',
 'CO',
 'CO2',
 'FCN',
 'H2CN',
 'HBO',
 'HCCF',
 'HCN',
 'HNC',
 'LiBH4',
 'NaCl',
 'NH3',
 'NH3O',
 'O3',
 'LiH',
 'BHF2',
 'C2H4',
 'CH3Cl',
 'Cl2',
 'ClCN',
 'CS',
 'CSO',
 'HBS',
 'HCCCl',
 'HCHS',
 'HCP',
 'Li',
 'Li2',
 'LiCl',
 'LiCN',
 'NaH',
 'NaLi',
 'NOCl',
 'NP',
 'OCl2',
 'OF2',
 'P2',
 'PH2',
 'PH3',
 'PH3O',
 'S2',
 'SCl2',
 'SH2',
 'SiH3',
 'SiH3Cl',
 'SiH3F',
 'SiH4',
 'SO2',
 'SO-trip']

In [28]:
len(get_functionals())

32

In [39]:
save(data)

b'\x1bSaving updates to GitHub...\x1b\n[master 7dfc79d] save Tue, Aug 18, 2020  1:39:58 PM\n 920 files changed, 885563 insertions(+), 20622 deletions(-)\n create mode 100644 output/Na2_B97M-V_aug-pc-4p.out\n create mode 100644 output/Na_M06-2X_aug-pc-4p.out\n create mode 100644 output/OF2_ccsdT_aug-cc-pcVTZp.err\n create mode 100644 output/cbs_limit/20-08-03/BH2F_ccsdT_aug-cc-pcVQZp.out\n create mode 100644 output/cbs_limit/20-08-03/CH2-t_ccsdT_aug-cc-pcV5Zp.out\n create mode 100644 output/cbs_limit/20-08-03/CH2BH_ccsdT_aug-cc-pcVTZp.out\n create mode 100644 output/cbs_limit/20-08-03/CN_ccsdT_aug-cc-pcV5Zp.out\n create mode 100644 output/cbs_limit/20-08-03/CO2_ccsdT_aug-cc-pcVQZp.out\n create mode 100644 output/cbs_limit/20-08-03/ClF_ccsdT_aug-cc-pcV5Zp.out\n create mode 100644 output/cbs_limit/20-08-03/ClF_ccsdT_aug-cc-pcVQZp.out\n create mode 100644 output/cbs_limit/20-08-03/F2_ccsdT_aug-cc-pcV5Zp.out\n create mode 100644 output/cbs_limit/20-08-03/FCN_ccsdT_aug-cc-pcVQZp.out\n create